In [1]:
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(
    context="talk",
    style="whitegrid",
    palette="colorblind",
    color_codes=True,
    rc={"figure.dpi": 150},
)

In [2]:
import yfinance as yf

data = yf.Ticker("AAPL").history(start="1996-01-01", end="2026-10-27")
data.index = data.index.tz_localize(None)
data = data.reset_index()
data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1996-01-02,0.241146,0.241146,0.237408,0.240211,139294400,0.0,0.0
1,1996-01-03,0.239277,0.245820,0.238342,0.240211,429833600,0.0,0.0
2,1996-01-04,0.242082,0.242082,0.234604,0.236006,300182400,0.0,0.0
3,1996-01-05,0.236473,0.256102,0.234604,0.256102,445928000,0.0,0.0
4,1996-01-08,0.257971,0.265448,0.254231,0.258905,121340800,0.0,0.0


In [3]:
import pandas as pd

df_cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL")
df_cpi = df_cpi.reset_index(drop=True)
df_cpi.head()

,observation_date,CPIAUCSL
0,1947-01-01,21.48
1,1947-02-01,21.62
2,1947-03-01,22.00
3,1947-04-01,22.00
4,1947-05-01,21.95


In [4]:
data["Date"] = pd.to_datetime(data["Date"])
monthly = data.set_index("Date")["Close"].resample("MS").last().to_frame("Close")

cpi = df_cpi.rename(columns={"observation_date": "date", "CPIAUCSL": "cpi"})
cpi["date"] = pd.to_datetime(cpi["date"])
cpi = cpi.set_index("date")

merged = monthly.join(cpi, how="inner")
merged = merged.reset_index()
merged.head()

,Date,Close,cpi
0,1996-01-01,0.206564,154.7
1,1996-02-01,0.205629,155.0
2,1996-03-01,0.183664,155.5
3,1996-04-01,0.182262,156.1
4,1996-05-01,0.195347,156.4


In [5]:
import numpy as np

merged["simple_return"] = merged["Close"].pct_change()
merged["log_return"] = np.log(merged["Close"] / merged["Close"].shift(1))
merged["inflation"] = merged["cpi"].pct_change()
merged["real_return"] = (1 + merged["simple_return"]) / (1 + merged["inflation"]) - 1

merged.head()

,Date,Close,cpi,simple_return,log_return,inflation,real_return
0,1996-01-01,0.206564,154.7,NaN,NaN,NaN,NaN
1,1996-02-01,0.205629,155.0,-0.004525,-0.004535,0.001939,-0.006451
2,1996-03-01,0.183664,155.5,-0.106819,-0.112966,0.003226,-0.109691
3,1996-04-01,0.182262,156.1,-0.007633,-0.007663,0.003859,-0.011448
4,1996-05-01,0.195347,156.4,0.071795,0.069335,0.001922,0.069739
